# Viviendas de Ames, Iowa: estimar el precio de venta

Este cuaderno selecciona algunas de las columnas más interpretables de las 80 disponibles, limpia el conjunto, explora los precios y compara modelos de regresión.

**En sencillo:** usamos la superficie, calidad, año de construcción, garaje y otras características para estimar el precio de venta de una casa. Son precios históricos de Ames, Iowa; no una tasación actual.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from ml_al_alfa.data import clean_ames_data, feature_defaults
from ml_al_alfa.datasets import load_ames_frame
from ml_al_alfa.train import ARTIFACTS_DIR, RAW_DIR, train_and_save

sns.set_theme(style="whitegrid")

## 1. Cargar y revisar los datos

Si el CSV no está ya en `data/raw/`, el proyecto lo descarga de su fuente pública.

In [ ]:
raw = load_ames_frame(RAW_DIR)
print(f"Filas y columnas originales: {raw.shape}")
display(raw.head())
print(f"Duplicados exactos: {raw.duplicated().sum()}")
display(raw.isna().sum().sort_values(ascending=False).head(12).rename("ausentes").to_frame())
print(f"Número de columnas originales: {raw.shape[1]}")

## 2. Seleccionar variables y limpiar

Usamos un subconjunto pequeño de columnas disponibles antes de la venta. Quitamos identificadores que no describen la casa y filas sin precio válido. El pipeline de entrenamiento completa las características incompletas usando solo datos de entrenamiento.

In [ ]:
X, y, cleaning_report = clean_ames_data(raw)
display(pd.Series(cleaning_report.as_dict(), name="recuento").to_frame())
display(X.head())
print(f"Precio mínimo y máximo: ${y.min():,.0f} – ${y.max():,.0f} USD")
print(f"Características seleccionadas: {len(X.columns)} de {raw.shape[1]}")

## 3. Explorar los precios y las características

El precio suele tener una distribución asimétrica. Las relaciones de los gráficos son descriptivas y no implican que una variable, por sí sola, determine el precio.

In [ ]:
plot_data = X.copy()
plot_data["SalePrice"] = y
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(plot_data["SalePrice"], bins=40, ax=axes[0])
axes[0].set(title="Distribución de precios", xlabel="Precio de venta (USD)")
sns.scatterplot(data=plot_data, x="Gr Liv Area", y="SalePrice", alpha=0.55, ax=axes[1])
axes[1].set(title="Superficie y precio", xlabel="Superficie habitable (pies²)", ylabel="Precio (USD)")
sns.boxplot(data=plot_data, x="Overall Qual", y="SalePrice", ax=axes[2])
axes[2].set(title="Precio por calidad general", xlabel="Calidad (1–10)", ylabel="Precio (USD)")
plt.tight_layout()
plt.show()

## 4. Entrenar y evaluar

El test contiene el 20% de las ventas separado al azar con `random_state=42`. El MAE se expresa en dólares: por ejemplo, un MAE de 20.000 USD significa que el error absoluto medio en ese test fue de unos 20.000 dólares.

In [ ]:
bundle = train_and_save(
    X,
    y,
    dataset_name="ames",
    rows=cleaning_report.as_dict(),
    output_path=ARTIFACTS_DIR / "ames_model.joblib",
)
metrics = pd.DataFrame(bundle["metrics"]).T.sort_values("mae")
display(metrics.rename(columns={"mae": "MAE (USD, menor es mejor)", "r2": "R²"}))
print(f"Mejor modelo: {bundle['model_name']}")
print(f"Modelo para Streamlit: {ARTIFACTS_DIR / 'ames_model.joblib'}")

## 5. Probar una estimación

El ejemplo usa valores típicos de la muestra limpia. Prueba el formulario de Streamlit para seleccionar otro tamaño, calidad o barrio.

In [ ]:
sample = pd.DataFrame([feature_defaults(X)], columns=bundle["features"])
estimated_price = float(bundle["pipeline"].predict(sample)[0])
display(sample)
print(f"Precio de venta estimado: ${estimated_price:,.0f} USD")
print(f"MAE de prueba: ${bundle['metrics'][bundle['model_name']]['mae']:,.0f} USD")